
# Determining Limiting Reactant with SMT Solvers

In this notebook, we will
look at SMT solvers and see how they can be used to formalize and solve
systems of equations. Particularly, we will show how we can determine the limiting reactant in a reaction. 
We will write a program that can do this to introduce you to Z3, a state-of-the-art SMT solver,
and its capabilities.


**Instructions:**
1. To get started, click on File on the top left and click "Save a copy in Drive."
This will give you an editable version of this document that you can use.
2. If you press `CMD`+`Enter` it runs the cell, and if you press `Shift`+`Enter` it runs the cell and goes to the next one.
3. Make sure you run all cells as you go through the notebook; some cells will not work properly unless the previous one
has been run too.
4. If you disconnect or are inactive for some time you should run all of the cells again.

## 0. Preliminaries (you should run this cell but there is no need to read it)

In [ ]:
!pip install z3-solver
!pip install git+https://github.com/crrivero/FormalMethodsTasting.git#subdirectory=core
from z3 import *
from tofmcore import showSolver
from IPython.display import clear_output
clear_output()

## Encoding constraints in Z3

The goal of this notebook is to teach you about formal methods;
particularly, how you can use existing formal verification tools
(in this case, Z3) to analyze and solve your own problems.
Before we get started, let's look at some basic things we can do with Z3.

### Integers

Let's use Z3 to solve problems involving integers. Let's start with something simple: find $x$ such that

$$2x + 5 = 15$$

In [ ]:
# Initialize variables

x = Int('x') # declairing that x is an integer named 'x'

# Initialize Z3 solver
s = Solver()

s.add( 2*x + 5 == 15 ) # add the equation

print(s)
print(s.check())
print(s.model())

Now let's try to check whether that's the only solution. We can do this by adding the following constraint to the solver:

$$x \not= 5$$

If the solver returns "**unsat**" then $x=5$ is the only solution.
Try it yourself by completing the code in the cell below.

In [ ]:
s.add( x == 5 ) # REPLACE THIS LINE
s.check()

## Determining Limiting Reactants using Z3

For the rest of this notebook, we will walk through solving a limiting reactants problem with Z3.

The reaction we will study is the combustion of propane. The balanced reaction is:

$$ C_3H_8 + 5O_2 \rightarrow 3CO_2 + 4H_2O $$

We will determine the limiting reactant when given 2 moles of propane and 5 moles of oxygen.

To model the problem, we must create variables to represent three things: The starting amount of each reactant in moles, the amount of each reactant used, and the amount of products produced.


In [ ]:
 # Initialize Solver
s = Solver()

# 1) Variables to represent the initial amount of propane and oxygen
initial_C3H8 = 2
initial_O2 = 8

# 2) Variables to represent the amount of propane and oxygen consumed
consumed_C3H8 = Real('C_{C_3H_8}')
consumed_O2 = Real('C_{O_2}')

# 3) Variables to represent the final amount of CO2 and H2O
produced_CO2 = Real('P_{CO_2}')
produced_H2O = Real('P_{H_2O}')

Lastly, we will use a variable to represent the "extent" of the reaction. 
This represents how much of the reaction can be completed with the given amounts.

For example, in our reaction, if we were given one mole of $C_3H_8$ and five moles $O_2$, the full reaction can completed once, so the extent is one.

In [ ]:
extent = Real('X')

Great! Now we need to do is add constraints to the solver.

First, we must determine the amount of reactants used, and products created, based on the extent of the reaction.

**Replace the lines below** to implement these constraints.

In [ ]:
# The amount of each reactant used depends on the extent of the reaction. We multiply the extent by the stoichiometric coefficient to find this.
s.add(consumed_C3H8 == extent * 1)
s.add(consumed_O2 == 0) # REPLACE THIS LINE

# We use a similar process to determine how much of each product was created, based on the extent of the reaction.
s.add(produced_CO2 == extent * 3)
s.add(produced_H2O == 0) # REPLACE THIS LINE

Finally, the last thing to do is add the limiting reactant logic.
The amount of reactants used cannot be more than what we are initially given.

**Replace the line below** to implement this in Z3.

In [ ]:
# Limiting Reactant Logic
s.add(consumed_C3H8 <= initial_C3H8)
s.add(False) # REPLACE THIS LINE

# Let's view our solver so far
showSolver(s)

Great! We have completed our model of the problem. The last thing to do is determine how far the reaction can go.

In our case, we need to do more than just find values of the variables that satisfy the equations.
We want to know how far the reaction is able to go until we run out of either reactant. 
In other words, we want to know what the *maximum* value for the extent is.

To do this, we will use Z3's Optimizer. This class is very similar to solver, as it also requires a set of variables and constraints. 
However, instead of finding any correct value for each variable, it allows us to find the maximum or minimum values of certain variables.

Run the cell below to create the Optimizer.

In [ ]:
# Initialize the Optimizer
opt = Optimize()

# Copy the variables and constraints from our solver
opt.add(s.assertions())

# Tell the optimizer to maximize the extent of the reaction
opt.maximize(extent)

In [ ]:
print( opt.check() ) # check if solution exists

In [ ]:
print( opt.model() ) # output solution


### Great Work! You have written a program to solve limiting reactant problems using Z3!


####If you'd like to continue your Z3 journey, you can start with this guide to learn more:
https://ericpony.github.io/z3py-tutorial/guide-examples.htm